## Limpieza de 'DF_BUSCAMETAS_SUCIO.csv'

Cargamos la tabla de carreras/modalidades (una fila por cada modalidad de cada evento) y le hacemos una primera inspección antes de limpiarla.

In [1]:
from pathlib import Path
import os

import pandas as pd


CSV_PATH = Path("../../data/raw/buscametas/DF_BUSCAMETAS_SUCIO.csv")
curses = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (329, 28)

DES_d                  float64
DES_desconegut         float64
DES_h                  float64
DES_total              float64
NP_d                   float64
NP_desconegut          float64
NP_h                   float64
NP_total               float64
RET_d                  float64
RET_desconegut         float64
RET_h                  float64
RET_total              float64
codigo                   int64
comarca_provincia       object
data                    object
event_id                 int64
finished_d             float64
finished_desconegut    float64
finished_h             float64
finished_total         float64
modalitat_nom           object
municipi                object
nom_cursa               object
sense_temps_d          float64
sense_temps_h          float64
sense_temps_total      float64
total_classificats       int64
total_inscrits           int64
dtype: object


,DES_d,DES_desconegut,DES_h,DES_total,NP_d,NP_desconegut,NP_h,NP_total,RET_d,RET_desconegut,...,finished_h,finished_total,modalitat_nom,municipi,nom_cursa,sense_temps_d,sense_temps_h,sense_temps_total,total_classificats,total_inscrits
0,NaN,NaN,NaN,NaN,8.0,NaN,37.0,45.0,NaN,NaN,...,147.0,164.0,ARETA TRAIL,Areta,Areta Trail Lasterketa,NaN,NaN,NaN,164,215
1,NaN,NaN,NaN,NaN,NaN,NaN,6.0,6.0,2.0,NaN,...,30.0,30.0,Epic Gravel,Ezcaray,Epic Gravel Ezcaray,NaN,NaN,NaN,34,45
2,NaN,NaN,NaN,NaN,NaN,NaN,6.0,6.0,NaN,NaN,...,38.0,40.0,Cicloturista 93K,Ezcaray,Epic Gravel Ezcaray,NaN,NaN,NaN,41,53
3,NaN,NaN,NaN,NaN,NaN,NaN,3.0,3.0,NaN,NaN,...,31.0,33.0,Cicloturista 57K,Ezcaray,Epic Gravel Ezcaray,NaN,NaN,NaN,33,36
4,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,NaN,NaN,...,10.0,12.0,Cicloturista 42K,Ezcaray,Epic Gravel Ezcaray,NaN,NaN,NaN,12,16


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados y rangos raros
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print("Filas duplicadas por (event_id, codigo):",
      curses.duplicated(subset=["event_id", "codigo"]).sum())
print()

print("Rango de fechas (texto):", curses["data"].min(), "->", curses["data"].max())
print()

print("total_inscrits <= 0 o nulo:", (curses["total_inscrits"].fillna(0) <= 0).sum())
print(curses["total_inscrits"].describe())
print()

print("total_classificats vs finished_total deberían coincidir:")
print((curses["total_classificats"] != curses["finished_total"]).sum(), "filas no coinciden")
print()

print("columnas de estado encontradas:", [c for c in curses.columns if c.endswith("_total")])

Valores nulos por columna:
DES_d                  306
DES_desconegut         327
DES_h                  308
DES_total              293
NP_d                   151
NP_desconegut          323
NP_h                   138
NP_total               122
RET_d                  231
RET_desconegut         323
RET_h                  212
RET_total              183
codigo                   0
comarca_provincia        0
data                     0
event_id                 0
finished_d              50
finished_desconegut    318
finished_h              35
finished_total          17
modalitat_nom            0
municipi                12
nom_cursa                0
sense_temps_d          235
sense_temps_h          219
sense_temps_total      212
total_classificats       0
total_inscrits           0
dtype: int64

Filas completamente duplicadas: 15
Filas duplicadas por (event_id, codigo): 15

Rango de fechas (texto): 2020-12-01 -> 2026-07-25

total_inscrits <= 0 o nulo: 0
count     329.000000
mean      228.598784


In [3]:
# Limpieza: nos quedamos solo con las columnas que interesan, renombradas.
# Aquí no hay distancia (esta fuente no la ofrece, a diferencia de
# carreirasgalegas), así que no la incluimos todavía (se extrae más abajo).
# finisher_d/finisher_h son el equivalente a ok_d/ok_h en otras fuentes:
# clasificados con tiempo registrado, por género (M->h, F->d, ver
# _sexo_sufix del scraper) — renombrados desde finished_d/finished_h para
# usar el mismo nombre de columna en las 4 fuentes.
curses_limpio = curses[
    ["codigo", "nom_cursa", "data", "municipi", "comarca_provincia", "modalitat_nom", "finished_d", "finished_h", "finished_desconegut"]
].rename(columns={
    "codigo": "id",
    "nom_cursa": "nombre_carrera",
    "data": "fecha",
    "municipi": "municipio",
    "comarca_provincia": "provincia",
    "modalitat_nom": "modalidad",
    "finished_d": "finisher_d",
    "finished_h": "finisher_h",
    "finished_desconegut": "finisher_desconegut",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])
curses_limpio[["finisher_d", "finisher_h"]] = curses_limpio[["finisher_d", "finisher_h"]].fillna(0).astype(int)

print(curses_limpio.shape)
curses_limpio.sample(25)

(329, 9)


,id,nombre_carrera,fecha,municipio,provincia,modalidad,finisher_d,finisher_h,finisher_desconegut
165,1207181251,Burdin Hesiko Mendi Lasterketa,2025-06-21,Gamiz-Fika,Bizkaia,Trail Largo,28,192,NaN
235,34,Oposiciones Bomberos Bilbao,2024-12-09,Bilbao,Bizkaia,Bomberos Bilbao,0,6,NaN
253,1206731756,Marcha Nórdica Villa de Azagra,2024-10-27,Azagra,Navarra,Federados,9,16,NaN
299,14,BTT Extreme Valle de Ezcaray,2023-07-21,Ezcaray,La Rioja,GENERAL - INDIVIDUAL,4,67,NaN
141,55,BTT Extreme Ezcaray - Domingo,2025-07-27,Ezcaray,La Rioja,Individual,7,119,NaN
167,1207183235,Burdin Hesiko Mendi Lasterketa,2025-06-21,Gamiz-Fika,Bizkaia,Martxa,103,90,NaN
33,129,Ugao-Miraballes Herri Krosa,2026-05-10,Ugao-Miraballes,Bizkaia,Aleví­n,19,28,NaN
156,1206142857,KV Valdezcaray,2025-07-19,Valdezcaray - Ezcaray,La Rioja,Cadete - Juvenil,1,4,NaN
26,161,Mitxarro Bira,2026-05-30,Araia,Álava,Trail Largo,8,59,NaN
234,1209870180,Soxoguti Erronka,2024-12-14,Artziniega,Álava,Carrera,42,150,NaN


In [4]:
# "modalidad" a menudo es en realidad la distancia (p.ej. "10K", "21.1K").
# Extraemos el número que va delante de la "K" — ya está en km, no hace
# falta pasarlo a metros. Con decimales (p.ej. "Nocturna 6,5K"), igual que
# en el resto de fuentes; si no hay match, 0.
distancia_extraida = curses_limpio["modalidad"].str.extract(r"(\d+(?:[.,]\d+)?)\s*[kK]")[0]
curses_limpio["distancia"] = distancia_extraida.str.replace(",", ".", regex=False).astype(float).fillna(0)

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de modalidad SIN 'nK' (distancia queda a 0) — revisa si hace falta cubrir algún caso más:")
print(curses_limpio.loc[curses_limpio["distancia"] == 0, "modalidad"].value_counts().head(30))
curses_limpio.head()

Filas con distancia detectada: 111 de 329

Ejemplos de modalidad SIN 'nK' (distancia queda a 0) — revisa si hace falta cubrir algún caso más:
modalidad
Trail Largo             20
Trail Corto             20
Mayores                 10
Infantil                 9
Benjamín                 9
Alevín                   7
Parejas                  6
Carrera                  6
Individual               5
Marcha Larga             4
Trail                    4
Prebenjamín              4
BTT                      4
Martxa                   4
GENERAL - INDIVIDUAL     3
Federados                3
GENERAL - PAREJAS        3
Marcha Corta             3
Bomberos Bilbao          3
Cadete                   3
Trekking Corto           2
Record del Kolitza       2
Promesas                 2
Trekking Largo           2
Absoluta Masculina       2
Con Discapacidad         2
No Federados             2
BTT Corto                2
BTT Largo                2
ARETA TRAIL              2
Name: count, dtype: int64


,id,nombre_carrera,fecha,municipio,provincia,modalidad,finisher_d,finisher_h,finisher_desconegut,distancia
0,197,Areta Trail Lasterketa,2026-07-25,Areta,Álava,ARETA TRAIL,17,147,NaN,0.0
1,198,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Epic Gravel,0,30,NaN,0.0
2,202,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista 93K,2,38,NaN,93.0
3,200,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista 57K,2,31,NaN,57.0
4,201,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista 42K,2,10,NaN,42.0


In [5]:
# Quitamos el "nK" de modalidad (ya está en distancia) y limpiamos los
# separadores que queden sueltos (comas, guiones, espacios de más).
curses_limpio["modalidad"] = (
    curses_limpio["modalidad"]
    .str.replace(r"\d+(?:[.,]\d+)?\s*[kK]\b", "", regex=True)
    .str.strip(" ,-")
    .str.replace(r"\s{2,}", " ", regex=True)
)

curses_limpio.head(20)

,id,nombre_carrera,fecha,municipio,provincia,modalidad,finisher_d,finisher_h,finisher_desconegut,distancia
0,197,Areta Trail Lasterketa,2026-07-25,Areta,Álava,ARETA TRAIL,17,147,NaN,0.0
1,198,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Epic Gravel,0,30,NaN,0.0
2,202,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista,2,38,NaN,93.0
3,200,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista,2,31,NaN,57.0
4,201,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,Cicloturista,2,10,NaN,42.0
5,203,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,BTT,3,10,NaN,0.0
6,204,Epic Gravel Ezcaray,2026-07-25,Ezcaray,La Rioja,E-Bike,0,4,NaN,0.0
7,196,KV Valdezcaray,2026-07-11,Valdezcaray - Ezcaray,La Rioja,KV Valdezcaray,19,75,NaN,0.0
8,195,KV Valdezcaray,2026-07-11,Valdezcaray - Ezcaray,La Rioja,Prebenjamin - Infantil,4,2,NaN,0.0
9,178,La Quadrako Mendi Lasterketa,2026-06-20,La Quadra - Güeñes,Bizkaia,Trail Largo,4,42,NaN,0.0


In [6]:
# Primera pasada: clasificamos modalidad en 5 categorías por palabras clave.
# Si "modalidad" no da ninguna pista (p.ej. "Individual", "Parejas",
# "Cadete"... son formato/edad, no disciplina), probamos con el nombre de
# la carrera antes de rendirnos a "Otros".
import re

_CATEGORIAS = {
    "trail running": r"trail|trekking|vertical|\btra\b|\bkv\b|\butpd\b",
    "Ciclismo y btt": r"btt|ciclis|bici|mtb|gravel|ciclotur|e-?bike|\bbike\b|gran ?fondo",
    "Multidisciplina": r"duatl|triatl|multidep|multidisci|aquatl|acuatl|combinada",
    "road running": r"carrera|running|popular|asfalto|ruta|marat|cross|absoluta|10k|5k|21k|half|corredor|lasterketa",
    "marcha":  r"martxa|marcha|\bmar\b|andarin",
}

def _clasificar_texto(texto):
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, texto):
            return categoria
    return None

def _clasificar(row):
    modalidad = row["modalidad"]
    texto_modalidad = "" if pd.isna(modalidad) else modalidad.lower().strip()
    categoria = _clasificar_texto(texto_modalidad) if texto_modalidad else None
    if categoria:
        return categoria

    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    categoria = _clasificar_texto(texto_nombre)
    if categoria:
        return categoria

    if texto_modalidad == "":
        # Ni la modalidad (vacía, era solo una distancia como "10K") ni el
        # nombre de la carrera dan ninguna pista — por defecto asumimos
        # road running, el formato más común en este sitio.
        return "road running"
    return "Otros"

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("Valores de 'modalidad' que han caído en 'Otros' (revisar si falta alguna palabra clave):")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "modalidad"].value_counts())

tipo_modalidad
trail running      105
Otros               73
road running        67
marcha              39
Ciclismo y btt      37
Multidisciplina      8
Name: count, dtype: int64

Valores de 'modalidad' que han caído en 'Otros' (revisar si falta alguna palabra clave):
modalidad
Mayores                     10
Infantil                     8
Benjamín                     7
Alevín                       5
Bomberos Bilbao              3
Vuelta San Cosme             2
Nocturna                     2
Promesas                     2
Record del Kolitza           2
Cadete                       2
Con Discapacidad             2
Prebenjamín                  2
10 años                      2
11 años                      2
12-13 años                   2
14-15 años                   2
Aleví­n                      2
Alevín FEM                   1
INFANTÍL                     1
MAYORES                      1
Prebenjamín FEM              1
Prebenjamín MAS              1
Benjamín FEM                 1
Benjamín

In [7]:
# Las modalidades que ya hemos clasificado por tipo (road running, trail
# running, Ciclismo y btt, Multidisciplina, marcha) ya han cumplido su
# función — las vaciamos para que solo quede texto sin clasificar (p.ej.
# "Infantil", "Benjamín"...) de cara a la clasificación de público que
# viene ahora.
curses_limpio.loc[curses_limpio["tipo_modalidad"] != "Otros", "modalidad"] = pd.NA

curses_limpio["modalidad"].isna().sum()

256

In [8]:
# Igual que con tipo_modalidad: clasificamos el público al que va dirigida
# la carrera (Mayores, Infantil, Benjamín...) por palabras clave. Antes
# comprobamos si es una carrera especial (con discapacidad, handbike...)
# o de élite/profesional — eso va a "Otros"/"Elite" porque no es una
# cuestión de edad. Si no hay ninguna marca de edad ni es una carrera
# especial, asumimos Absoluta/General por defecto (la inmensa mayoría de
# los casos sin categoría son justamente eso).
_OTROS_PATRON = r"discapacidad|invidente|handbike|silla de ruedas|cadeira de rodas"
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

_PUBLICOS = {
    "Elite": r"\belit|profesional",
    # Cadete/Juvenil se trata como la misma categoría que Infantil (ver
    # Limpieza_union.ipynb: en el resto de fuentes tampoco hay ese
    # desglose, así que "Infantil" significa lo mismo en las 10 fuentes).
    "Infantil": r"infant|benjam|alev|prebenjam|chupet|peque|años|cadete|juvenil|junior|cadet\b|promesa|preuniversitari",
    "Mayores/Veteranos": r"mayores|veteran|master|m[aá]ster",
}

def _clasificar_publico(row):
    # "Equipos" (carreras por equipos/relevos) manda por encima de
    # cualquier otra cosa, y se busca tanto en el nombre de la carrera
    # como en la modalidad — igual que en ccnorte.
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    if re.search(_EQUIPOS_PATRON, texto_nombre):
        return "Equipos"

    modalidad = row["modalidad"]
    if pd.isna(modalidad):
        return "Absoluta/General"
    texto = modalidad.lower()

    if re.search(_EQUIPOS_PATRON, texto):
        return "Equipos"

    if re.search(_OTROS_PATRON, texto):
        return "Otros"

    for publico, patron in _PUBLICOS.items():
        if re.search(patron, texto):
            return publico

    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Valores de 'modalidad' clasificados como Otros (carreras especiales):")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts())

publico
Absoluta/General     269
Infantil              47
Mayores/Veteranos     11
Otros                  2
Name: count, dtype: int64

Valores de 'modalidad' clasificados como Otros (carreras especiales):
modalidad
Con Discapacidad    2
Name: count, dtype: int64


### Comarca — geocodificada a partir de municipio/provincia

`municipio`/`provincia` ya vienen de la fuente, pero no `comarca`. La geocodificamos con Nominatim (igual que en carreirasgalegas/ccnorte/championsxip), usando `municipio, provincia, España` para desambiguar (aquí hay municipios de varias provincias: Álava, Bizkaia, La Rioja, Navarra, Cantabria, Burgos, Jaén, A Coruña...). `county` (comarca) en OpenStreetMap suele salir vacío en Euskadi/La Rioja — no es un fallo del código, es que esa etiqueta administrativa apenas está mapeada ahí.

In [9]:
import csv
import time


def geocodificar_comarcas(curses_limpio, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_com = out_path / "buscametas_comarcas.csv"

    pares = (
        curses_limpio[["municipio", "provincia"]]
        .drop_duplicates()
        .dropna(subset=["municipio"])
    )

    cache = {}
    if csv_com.exists():
        prev = pd.read_csv(csv_com, dtype=str)
        cache = {(r["municipio"], r["provincia"]): r["comarca"] for _, r in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="buscametas_comarcas_claudia")

    pendientes = [
        (m, p) for m, p in pares.itertuples(index=False)
        if (m, p) not in cache
    ]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(pares)} únicos)")

    write_header = not csv_com.exists()
    with open(csv_com, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["municipio", "provincia", "comarca"])
        if write_header:
            writer.writeheader()

        for i, (municipio, provincia) in enumerate(pendientes, 1):
            comarca = None
            try:
                query = f"{municipio}, {provincia}, España" if pd.notna(provincia) else f"{municipio}, España"
                loc = geolocator.geocode(
                    query, exactly_one=True, country_codes="es", addressdetails=True, timeout=10,
                )
                if loc:
                    comarca = loc.raw.get("address", {}).get("county")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow({"municipio": municipio, "provincia": provincia, "comarca": comarca})
            f.flush()
            cache[(municipio, provincia)] = comarca

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de comarcas: {csv_com.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/buscametas")
_comarcas = geocodificar_comarcas(curses_limpio, out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio.apply(
    lambda row: _comarcas.get((row["municipio"], row["provincia"])), axis=1
)

print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "provincia", "comarca"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 61 municipios ya geocodificados
Municipios a geocodificar: 0 (de 61 únicos)
CSV de comarcas: /Users/claudiarm2002/Desktop/TFM/data/raw/buscametas/buscametas_comarcas.csv
Filas con comarca: 40 de 329


,municipio,provincia,comarca
88,Orduña,Bizkaia,NaN
104,Galdames,Bizkaia,NaN
106,Ancín-Antzin,Navarra,Estellerria
213,Laudio,Álava,NaN
240,Erandio,Bizkaia,NaN
121,Pradoluengo,Burgos,NaN
28,Galdakao,Bizkaia,NaN
184,Zeberio,Bizkaia,NaN
7,Valdezcaray - Ezcaray,La Rioja,NaN
170,Ribafrecha,La Rioja,NaN


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, championsxip, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("buscametas") que identifica de qué origen viene cada fila, útil sobre todo al concatenar las 10 tablas en una sola. Lo que es propio solo de buscametas (`finisher_desconegut`, `id`) va al final.

In [10]:
# "modalidad" ya ha cumplido su función: la hemos reclasificado en
# tipo_modalidad y publico, así que la quitamos. Añadimos "fuente"
# (constante, para identificar el origen al concatenar con las otras 5
# tablas) y "dia_semana" (derivado de "fecha"), y reordenamos las columnas
# para que el esquema común (fuente, nombre_carrera, fecha, dia_semana,
# distancia, tipo_modalidad, publico, finisher_d, finisher_h, municipio,
# comarca, provincia) quede igual en las 6 fuentes, dejando lo propio de
# buscametas (finisher_desconegut, id) al final.
curses_limpio = curses_limpio.drop(columns=["modalidad"])

curses_limpio["fuente"] = "buscametas"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconegut", "id"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconegut',
 'id']

In [11]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia', 'finisher_desconegut', 'id']
Filas x columnas: (329, 14)

fuente                         object
nombre_carrera                 object
fecha                  datetime64[ns]
dia_semana                     object
distancia                     float64
tipo_modalidad                 object
publico                        object
finisher_d                      int64
finisher_h                      int64
municipio                      object
comarca                        object
provincia                      object
finisher_desconegut           float64
id                              int64
dtype: object

Cruce tipo_modalidad x publico:
publico          Absoluta/General  Infantil  Mayores/Veteranos  Otros
tipo_modalidad                                                       
Ciclismo y btt                 37         0           

,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,finisher_desconegut,id
297,buscametas,BTT Extreme Valle de Ezcaray - Domingo,2024-07-28,Domingo,0.0,Ciclismo y btt,Absoluta/General,7,90,Ezcaray,NaN,La Rioja,NaN,16
307,buscametas,EKOC | Güeñes - Gordexola,2021-01-08,Viernes,19.0,trail running,Absoluta/General,3,43,Güeñes,NaN,Bizkaia,NaN,1137718597
97,buscametas,Record Kolitza Internacional,2025-11-01,Sábado,0.0,Otros,Absoluta/General,1,1,Balmaseda,NaN,Bizkaia,NaN,73
58,buscametas,Apuko Igoera Zaramillo Enkarterri,2026-02-22,Domingo,30.0,road running,Absoluta/General,25,216,Zaramillo,NaN,Bizkaia,NaN,87
136,buscametas,Picos de La Demanda,2025-09-13,Sábado,24.0,road running,Absoluta/General,38,211,Ezcaray,NaN,La Rioja,NaN,1209475470
60,buscametas,Pruebas Crono,2026-02-12,Jueves,30.0,road running,Absoluta/General,0,0,Zaramillo,NaN,Bizkaia,NaN,89
211,buscametas,Lau Haizeta Trail,2025-03-23,Domingo,0.0,trail running,Absoluta/General,60,148,Mungia,NaN,Bizkaia,NaN,1202571742
9,buscametas,La Quadrako Mendi Lasterketa,2026-06-20,Sábado,0.0,trail running,Absoluta/General,4,42,La Quadra - Güeñes,NaN,Bizkaia,NaN,178
167,buscametas,Burdin Hesiko Mendi Lasterketa,2025-06-21,Sábado,0.0,marcha,Absoluta/General,103,90,Gamiz-Fika,NaN,Bizkaia,NaN,1207183235
309,buscametas,EKOC | Güeñes - Gordexola,2021-01-08,Viernes,39.0,Ciclismo y btt,Absoluta/General,3,119,Güeñes,NaN,Bizkaia,NaN,1137720347


In [12]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/buscametas/DF_BUSCAMETAS_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en ../../data/processed/buscametas/DF_BUSCAMETAS_LIMPIO.csv
